# Phase 1 — Baseline upscaler

Runs four methods (bicubic, Lanczos, Real-ESRGAN x4plus, stable-diffusion-x4-upscaler) on the 60-image frozen test set at 4× / 5× / 10×. Produces per-slice contact sheets in `outputs/phase1/`.

**Runtime:** ~20 min first run, dominated by the diffusion pipeline. Upscaled outputs are cached to `outputs/phase1/upscales/` so re-runs are fast.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from tqdm.auto import tqdm

from upscaler import baselines, pipeline, testset

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PHASE1_DIR = REPO_ROOT / "outputs" / "phase1"
CACHE_DIR = PHASE1_DIR / "upscales"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__}, cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
print(f"outputs -> {PHASE1_DIR}")

In [ ]:
ts = testset.load(REPO_ROOT / "data" / "test_images")
print(
    f"loaded {len(ts)} images: {len(ts.slice(category='traditional'))} traditional + {len(ts.slice(category='hard'))} hard"
)
print(f"subcategories: {ts.subcategories}")
print(f"challenges:    {ts.challenges}")

In [ ]:
METHODS = ("bicubic", "lanczos", "realesrgan", "x4_upscaler")
LR_SIZES = (100, 200, 250)
TARGET = 1000  # canonical output size


def run_method(
    method: str, lr_img: Image.Image, scale: float, up_pipe: pipeline.UpscalerPipeline
) -> Image.Image:
    if method == "bicubic":
        return baselines.bicubic(lr_img, scale)
    if method == "lanczos":
        return baselines.lanczos(lr_img, scale)
    if method == "realesrgan":
        return baselines.realesrgan(lr_img, scale)
    if method == "x4_upscaler":
        # x4-upscaler does its native 4x; we bicubic-fit to the requested target so
        # all four methods produce comparable 1000x1000 outputs for the contact sheets.
        out = up_pipe.upscale_x4(lr_img)
        target_w = round(lr_img.size[0] * scale)
        if out.size != (target_w, target_w):
            out = out.resize((target_w, target_w), Image.Resampling.BICUBIC)
        return out
    raise ValueError(f"unknown method: {method}")


def cached_upscale(img_name: str, lr_size: int, method: str, up_pipe) -> Path:
    out_path = CACHE_DIR / f"{Path(img_name).stem}_{lr_size}_{method}.jpg"
    if out_path.is_file():
        return out_path
    lr = Image.open(ts.root / f"{Path(img_name).stem}_{lr_size}.jpg").convert("RGB")
    scale = TARGET / lr_size
    out = run_method(method, lr, scale, up_pipe)
    if out.size != (TARGET, TARGET):
        out = out.resize((TARGET, TARGET), Image.Resampling.BICUBIC)
    out.save(out_path, quality=92)
    return out_path

In [ ]:
# 60 images x 3 LR sizes x 4 methods = 720 upscales. Bicubic/Lanczos are
# effectively free; Real-ESRGAN is ~0.3s each; x4-upscaler dominates (~5-8s).
up_pipe = pipeline.UpscalerPipeline()

work = [(img.name, lr, method) for img in ts for lr in LR_SIZES for method in METHODS]

for name, lr, method in tqdm(work, desc="upscaling"):
    cached_upscale(name, lr, method, up_pipe)

up_pipe.close()
print(f"\n{len(list(CACHE_DIR.glob('*.jpg')))} upscales cached in {CACHE_DIR}")

In [ ]:
def render_contact_sheet(
    images: list,
    title: str,
    lr_size: int,
    out_path: Path,
    crop_size: int = 400,
) -> None:
    """Render a slice contact sheet: one row per image, columns = HR | bicubic | Lanczos | Real-ESRGAN | x4-upscaler.

    Cells show a center crop so pixel-level differences are visible on the rendered sheet.
    """
    cols = ["HR (ref)", "bicubic", "Lanczos", "Real-ESRGAN", "x4-upscaler"]
    n_rows = len(images)
    if n_rows == 0:
        return

    fig, axes = plt.subplots(n_rows, len(cols), figsize=(3 * len(cols), 3 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    fig.suptitle(
        f"{title}  —  LR {lr_size}→{TARGET}  (center {crop_size}×{crop_size} crop)", fontsize=14
    )

    cx = (TARGET - crop_size) // 2
    crop_box = (cx, cx, cx + crop_size, cx + crop_size)

    for r, img in enumerate(images):
        hr = Image.open(img.hr_path).convert("RGB").crop(crop_box)
        cells = [hr]
        for method in ("bicubic", "lanczos", "realesrgan", "x4_upscaler"):
            p = CACHE_DIR / f"{Path(img.name).stem}_{lr_size}_{method}.jpg"
            cells.append(Image.open(p).convert("RGB").crop(crop_box))

        for c, (cell, col_name) in enumerate(zip(cells, cols, strict=True)):
            ax = axes[r, c]
            ax.imshow(cell)
            ax.axis("off")
            if r == 0:
                ax.set_title(col_name, fontsize=11)
            if c == 0:
                ax.text(
                    -0.05,
                    0.5,
                    Path(img.name).stem,
                    rotation=0,
                    ha="right",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=9,
                )

    fig.tight_layout(rect=[0.02, 0, 1, 0.98])
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)

In [ ]:
# Primary deliverable: one sheet per slice at each LR ratio.
# Traditional: sheets per subcategory.
# Hard: sheets per challenge tag.

slices: list[tuple[str, list]] = []
for sub in ts.subcategories:
    slices.append((f"traditional_{sub}", ts.slice(subcategory=sub)))
for challenge in ts.challenges:
    slices.append((f"hard_{challenge}", ts.slice(challenge=challenge)))

for slice_name, slice_images in tqdm(slices, desc="contact sheets"):
    for lr_size in LR_SIZES:
        ratio = TARGET // lr_size
        out_name = (
            f"{slice_name}_x{ratio}.png" if TARGET % lr_size == 0 else f"{slice_name}_{lr_size}.png"
        )
        render_contact_sheet(
            slice_images,
            title=slice_name.replace("_", " / "),
            lr_size=lr_size,
            out_path=PHASE1_DIR / out_name,
        )

print(f"\n{len(list(PHASE1_DIR.glob('*.png')))} contact sheets in {PHASE1_DIR}")

In [ ]:
# Preview: show one sheet inline so this notebook carries visible evidence.
preview = PHASE1_DIR / "traditional_landscape_x4.png"
if preview.is_file():
    from IPython.display import Image as IPImage
    from IPython.display import display

    display(IPImage(filename=str(preview)))
else:
    print(f"(preview {preview.name} not generated — check the contact-sheet cell above)")